In [1]:
# ============================================================
# Jupyter Notebook Code: Inference for 16 trained checkpoints
# Rows   : Model
# Columns: Evaluation vehicle
# ============================================================

# =========================
# Cell 1: Imports
# =========================
import os
import gc
import glob
import random
import warnings
import numpy as np
import pandas as pd
import torch

from dotenv import load_dotenv
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
)

from utils import SegmentFromFile

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

/home/lisa/Arupreza/UIDS-II/uids/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =========================
# Cell 2: Environment + Seed
# =========================
load_dotenv()
hf_token = os.getenv("HF_TOKEN", None)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# =========================
# Cell 3: Configuration
# One trained checkpoint for each (Model, Vehicle) pair
# =========================
CHECKPOINTS = {
    "BERT": {
        "Kia Soul": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/Bert/TrainKia",
        "Tesla Model 3": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/Bert/TrainTesla",
        "Genesis G80": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/Bert/TrainGen",
        "Chevrolet Silverado": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/Bert/TrainSil",
    },
    "ALBERT": {
        "Kia Soul": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/AlBert/TrainKia",
        "Tesla Model 3": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/AlBert/TrainTesla",
        "Genesis G80": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/AlBert/TrainGen",
        "Chevrolet Silverado": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/AlBert/TrainSil",
    },
    "TinyBERT": {
        "Kia Soul": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/TinyBert/TrainKia",
        "Tesla Model 3": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/TinyBert/TrainTesla",
        "Genesis G80": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/TinyBert/TrainGen",
        "Chevrolet Silverado": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/TinyBert/TrainSil",
    },
    "MobileBERT": {
        "Kia Soul": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/MobileBert/TrainKia/mobilebert-can-attack-classifier-experimental",
        "Tesla Model 3": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/MobileBert/TrainTesla/mobilebert-can-attack-classifier-experimental",
        "Genesis G80": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/MobileBert/TrainGen/mobilebert-can-attack-classifier-experimental",
        "Chevrolet Silverado": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/MobileBert/TrainSil/mobilebert-can-attack-classifier-experimental",
    },
}

BASE_MODELS = {
    "BERT": "google-bert/bert-base-uncased",
    "ALBERT": "albert/albert-base-v2",
    "TinyBERT": "nreimers/TinyBERT_L-4_H-312_v2",
    "MobileBERT": "google/mobilebert-uncased",
}

EVAL_DIRS = {
    "All": "/home/lisa/Arupreza/UIDS/UIDS-II/Split_data/Val",
}

TIME_GAP_EVAL = 100.0
MAX_LENGTH = 512
OUTPUT_ROOT = "./inference_only_results_16"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

In [ ]:
# =========================
# Cell 4: Helper Functions
# =========================
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def format_chunk_to_string(chunk):
    tokens = []
    for pair in chunk:
        tokens.append(f"T{int(pair[0])}")
        tokens.append(f"G{int(pair[1])}")
    return " ".join(tokens)

def load_and_process_data(directory, time_gap):
    all_chunks, all_labels = [], []
    csv_files = sorted(glob.glob(os.path.join(directory, "*.csv")))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {directory}")

    for file_path in csv_files:
        filename = os.path.basename(file_path)
        chunks, labels = SegmentFromFile(directory, filename, time_gap=time_gap)
        all_chunks.extend(chunks)
        all_labels.extend(labels)

    texts = [format_chunk_to_string(chunk) for chunk in all_chunks]
    return pd.DataFrame({"text": texts, "label": all_labels})

def tokenize_dataset(df, tokenizer, max_length=512):
    ds = Dataset.from_pandas(df, preserve_index=False)

    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, padding=False, max_length=max_length)

    ds = ds.map(tokenize_function, batched=True)
    cols_to_remove = [c for c in ["text", "__index_level_0__"] if c in ds.column_names]
    if cols_to_remove:
        ds = ds.remove_columns(cols_to_remove)
    ds = ds.rename_column("label", "labels")
    ds.set_format("torch")
    return ds

def compute_metrics_from_arrays(y_true, y_pred):
    avg = "binary" if len(set(y_true)) == 2 else "weighted"
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=avg, zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)
    return {"F1": float(f1), "Accuracy": float(acc), "Precision": float(precision), "Recall": float(recall)}

def load_tokenizer(checkpoint_path, base_model_name, hf_token=None):
    if os.path.exists(os.path.join(checkpoint_path, "tokenizer_config.json")):
        return AutoTokenizer.from_pretrained(checkpoint_path, token=hf_token, use_fast=True)
    return AutoTokenizer.from_pretrained(base_model_name, token=hf_token, use_fast=True)

In [ ]:
# =========================
# Cell 5: Preload evaluation sets once
# =========================
eval_dataframes = {}
for vehicle, path in EVAL_DIRS.items():
    print("Loading eval set:", vehicle)
    eval_dataframes[vehicle] = load_and_process_data(path, TIME_GAP_EVAL)

In [ ]:
# =========================
# Cell 6: Inference Loop
# =========================
all_results = []

for model_name, vehicle_ckpts in CHECKPOINTS.items():
    base_model = BASE_MODELS[model_name]

    for vehicle_name, checkpoint_path in vehicle_ckpts.items():
        print(f"\nRunning {model_name} on {vehicle_name}")

        clear_memory()

        eval_df = eval_dataframes[vehicle_name]
        tokenizer = load_tokenizer(checkpoint_path, base_model, hf_token)
        eval_dataset = tokenize_dataset(eval_df, tokenizer, MAX_LENGTH)
        data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

        model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path, token=hf_token)

        args = TrainingArguments(
            output_dir=os.path.join(OUTPUT_ROOT, f"tmp_{model_name}_{vehicle_name.replace(' ', '_')}"),
            per_device_eval_batch_size=8,
            report_to="none",
            fp16=torch.cuda.is_available(),
        )

        trainer = Trainer(
            model=model,
            args=args,
            tokenizer=tokenizer,
            data_collator=data_collator,
        )

        pred = trainer.predict(eval_dataset)
        y_true = pred.label_ids
        y_pred = pred.predictions.argmax(-1)
        metrics = compute_metrics_from_arrays(y_true, y_pred)

        all_results.append({
            "Model": model_name,
            "Vehicle": vehicle_name,
            "F1": metrics["F1"],
            "Accuracy": metrics["Accuracy"],
            "Precision": metrics["Precision"],
            "Recall": metrics["Recall"],
            "Samples": len(eval_df),
        })

        del trainer, model, tokenizer
        clear_memory()

In [ ]:
# =========================
# Cell 7: Final F1 Table
# =========================
results_df = pd.DataFrame(all_results)
pivot_f1 = results_df.pivot(index="Model", columns="Vehicle", values="F1")
pivot_f1 = pivot_f1[["Kia Soul", "Tesla Model 3", "Genesis G80", "Chevrolet Silverado"]]

print("Final F1 comparison table:")
display(pivot_f1.round(4))

pivot_f1.round(4).to_csv(os.path.join(OUTPUT_ROOT, "final_f1_comparison_table.csv"))
print("Saved:", os.path.join(OUTPUT_ROOT, "final_f1_comparison_table.csv"))